# Pydantic for AI Agents
- Notebook by Adam Lang
- Date: 9-12-2026

# Why use Pydantic with LangGraph?

1. **Data Validation & Parsing**
   - Defines how data should be structured using standard Python type checks, automatically enforcing these rules.
   - LLM gives output in unstructured data --> Pydantic validates these outputs

2. **Type Hint Integration**
   - Python type annotations are used to define schemas.
   - This reduces the need for verbose validation code.

3. **Fast Performance**
   - Core validation engine written in RUST -- VERY FAST!

4. **Strict and Lax Modes**
   - Supports BOTH strict mode (enforcing strict types) AND lax mode (attempting to coerce data -- converting "1" to 1)

5. **Clear Error Handling**
   - Provides detailed errors when data validation fails.
  
6. **JSON Schema Generation**
   - Pydantic models can easily generate JSON schemas for documentation or validation in other languages.

## Problem without Pydantic

In [1]:
## example function without Pydantic
## Use Case: adding patient to database
def add_patient_data(name: str, age: int):
  ## type check
  if type(name) == str and type(age) == int:
    if age >= 0:
      print(name)
      print(age)
      print("Data added successfully to the database!")
    else:
      raise ValueError("Age cannot be negative.")

  else:
    raise TypeError("Invalid data type for name or age. Name shold be a string and age should be an integer.")


## another function -- add patient_data
def update_patient_data(name:str, age: int):
  if type(name) == str and type(age) == int:
    if age >= 0:
      print(name)
      print(age)
      print("Data added successfully to the database!")

    else:
      raise ValueError("Age cannot be negative.")

  else:
    raise TypeError("Invalid data type for name or age. Name should be string, Age should be Integer.")

In [2]:
## call function
add_patient_data("Joe", "twenty five")

TypeError: Invalid data type for name or age. Name shold be a string and age should be an integer.

### Summary
- We can see it errored above due to the type check we put on age for int.
- We can also see how inefficient this is because the code is less modular and we have to repeat the same type checks.

In [3]:
## but if we put this in correctly:
add_patient_data("Joe", -25)

ValueError: Age cannot be negative.

### Summary
- We can see above it also checked the age cannot be negative.

### Key Takeaway:
- Pydantic can be used to solve this problem. Otherwise a user can literally enter anything they want -- this is why we need data validation checks.

# How to Use Pydantic
1. **First Define a Pydantic Model** that represents the **ideal schema** of the data.
  - This includes the expected fields, their types, and any validation constraints (e.g. `gt=0` for positive numbers)

2. **Instantiate the model with raw input data** (usually a dictionary or JSON-like structure)
  - Pydantic will automatically validate the data and coerce it into the correct Python types (if possible).
  - If the data does not meet the model's requirements, Pydantic will raise a `ValidationError`.

3. **Pass the validated model object** to functions or use it throughout your codebase.
  - This ensures that every part of your program works with **clean, type-safe, and logically valid data.**

In [4]:
%pip install pydantic>=2.0.0

In [13]:
## imports
from pydantic import BaseModel

## 1. Pydantic model -->  pydantic class
class PatientData(BaseModel):
  name: str
  age: int
  weight: float


## add patient_data
def add_patient_data(patient: PatientData):
  print(patient.name)
  print(patient.age)
  print("Data added successfully to the database!")


## update patient_data
def update_patient_data(patient: PatientData):
  print(patient.name)
  print(patient.age)
  print("Data updated successfully in the database!")

## 2. Init model with raw input data
patient_data = {"name": "Joe", "age": 30, "weight": "70.5"}

## 3. Pass validated model object to functions
patient_1 = PatientData(**patient_data) ## ** will unpack the dict


## use patient functions
add_patient_data(patient_1)


Joe
30
Data added successfully to the database!


In [14]:
patient_data = {"name": "Julian", "age": "35", "weight": "70.5"}

## 3. Pass validated model object to functions
patient_2 = PatientData(**patient_data) ## ** will unpack the dict


## update patient data
update_patient_data(patient_2)

Julian
35
Data updated successfully in the database!


## Adding Complexity to Pydantic Models
- We can use the previous pydantic model and add some complexity to it.

In [21]:
## imports
from pydantic import BaseModel
from typing import List, Dict

## 1. Pydantic model -->  pydantic class (schema)
class PatientData(BaseModel):
  name: str
  age: int
  weight: float
  married: bool
  allergies: List[str] # list of strings
  contact_info: Dict[str, str] # dictionary of key:value pair strings


## add patient_data
def add_patient_data(patient: PatientData):
  print(patient.name)
  print(patient.age)
  print(patient.weight)
  print(patient.married)
  print(patient.allergies)
  print(patient.contact_info)
  print("Data added successfully to the database!")


## update patient_data
def update_patient_data(patient: PatientData):
  print(patient.name)
  print(patient.age)
  print(patient.weight)
  print(patient.married)
  print(patient.allergies)
  print(patient.contact_info)
  print("Data updated successfully in the database!")

## 2. Init model with raw input data
patient_data = {"name": "Joe",
                "age": "30",
                "weight": "70.5",
                "married": "True",
                "allergies": ["peanuts","shellfish"], # Corrected typo here
                "contact_info": {"email":"joe@gmail.com","address": "7 hill drive","phone_num":"123-456-7890"}}

## 3. Pass validated model object to functions
patient_1 = PatientData(**patient_data) ## ** will unpack the dict


## use patient functions
add_patient_data(patient_1)


Joe
30
70.5
True
['peanuts', 'shellfish']
{'email': 'joe@gmail.com', 'address': '7 hill drive', 'phone_num': '123-456-7890'}
Data added successfully to the database!


## Pydantic - Required and Optional Fields
- Optional allows us to make a field optional and still execute.

In [23]:
## imports
from pydantic import BaseModel
from typing import List, Dict, Optional

## 1. Pydantic model -->  pydantic class (schema)
class PatientData(BaseModel):
  name: str
  age: int
  weight: float
  married: bool = False
  allergies: Optional[List[str]] = None
  contact_info: Dict[str, str] # dictionary of key:value pair strings


## add patient_data
def add_patient_data(patient: PatientData):
  print(patient.name)
  print(patient.age)
  print(patient.weight)
  print(patient.married)
  print(patient.allergies)
  print(patient.contact_info)
  print("Data added successfully to the database!")


# ## update patient_data
# def update_patient_data(patient: PatientData):
#   print(patient.name)
#   print(patient.age)
#   print(patient.weight)
#   print(patient.married)
#   print(patient.allergies)
#   print(patient.contact_info)
#   print("Data updated successfully in the database!")

## 2. Init model with raw input data
patient_data = {"name": "Joe",
                "age": "30",
                "weight": "70.5",
                "married": "True",
                # "allergies": ["peanuts","shellfish"], # Corrected typo here
                "contact_info": {"email":"joe@gmail.com","address": "7 hill drive","phone_num":"123-456-7890"}}

## 3. Pass validated model object to functions
patient_1 = PatientData(**patient_data) ## ** will unpack the dict


## use patient functions
add_patient_data(patient_1)


Joe
30
70.5
True
None
{'email': 'joe@gmail.com', 'address': '7 hill drive', 'phone_num': '123-456-7890'}
Data added successfully to the database!


### Summary
- With these additional functions we can make certain fields optional and Required.

## Data Validation
- Data Validation allows you to be very granular with your field requirements.
- This is very helpful for data contracts/API handling where you know the specific data requirements for your application.

In [27]:
%pip install pydantic[email]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 27.7 MB/s eta 0:00:00


In [37]:
## imports
from pydantic import BaseModel, EmailStr, AnyUrl, Field
from typing import List, Dict, Optional

## 1. Pydantic model -->  pydantic class (schema)
class PatientData(BaseModel):
  name: str = Field(max_length=50)
  age: int = Field(gt=0, lt=100)
  email: EmailStr
  linkedin_url: AnyUrl
  weight: float
  married: bool = False
  allergies: Optional[List[str]] = Field(max_length=5)
  contact_info: Dict[str, str] # dictionary of key:value pair strings


## add patient_data
def add_patient_data(patient: PatientData):
  print(patient.name)
  print(patient.age)
  print(patient.weight)
  print(patient.married)
  print(patient.allergies)
  print(patient.contact_info)
  print("Data added successfully to the database!")


# ## update patient_data
# def update_patient_data(patient: PatientData):
#   print(patient.name)
#   print(patient.age)
#   print(patient.weight)
#   print(patient.married)
#   print(patient.allergies)
#   print(patient.contact_info)
#   print("Data updated successfully in the database!")

## 2. Init model with raw input data
patient_data = {"name": "Joe",
                "age": 25,
                "weight": "70.5",
                "email": "joe@gmail.com",
                "linkedin_url": "https://www.linkedin.com/in/joecool",
                "married": "True",
                "allergies": ["peanuts","shellfish"],
                "contact_info": {"email":"joe@gmail.com","address": "7 hill drive","phone_num":"123-456-7890"}}

## 3. Pass validated model object to functions
patient_1 = PatientData(**patient_data) ## ** will unpack the dict


## use patient functions
add_patient_data(patient_1)


Joe
25
70.5
True
['peanuts', 'shellfish']
{'email': 'joe@gmail.com', 'address': '7 hill drive', 'phone_num': '123-456-7890'}
Data added successfully to the database!


## Metadata Validation
- This is another example of using Field for validation (above we used it for schema).
- The key difference here is that we need to use the **Annotated** object to include field metadata

In [42]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field
from typing import List, Dict, Optional, Annotated

## patient data model
class PatientData(BaseModel):

  name: Annotated[str, Field(max_length=50, title="Name of the patient", description='Give the name of the patient in less than 50 chars', examples=['Joe','Amy'])]
  email: EmailStr
  linkedin_url: AnyUrl
  age: int = Field(gt=0, lt=120)
  weight: Annotated[float, Field(gt=0, strict=True, description='Weight of the patient in kg')]
  married: Annotated[bool, Field(default=None, description='Is the patient married or not')]
  allergies: Annotated[Optional[List[str]], Field(default=None, max_length=5)]
  contact_details: Dict[str, str]

def add_patient_data(patient: PatientData):
  print(patient.name)
  print(patient.age)
  print(patient.weight)
  print(patient.married)
  print(patient.allergies)
  print("Data added successfully to the database!")

patient_data = {"name": "Joe",
                "age": 25,
                "weight": 70.5,
                "email": "joe@gmail.com",
                "linkedin_url": "https://www.linkedin.com/in/joecool",
                "married": "True",
                "allergies": ["peanuts","shellfish"],
                "contact_details": {"email":"joe@gmail.com","address": "7 hill drive","phone_num":"123-456-7890"}}

## 3. Pass validated model object to functions
patient_1 = PatientData(**patient_data) ## ** will unpack the dict


## use patient functions
add_patient_data(patient_1)


Joe
25
70.5
True
['peanuts', 'shellfish']
Data added successfully to the database!


## Field Validator in Pydantic
- This lets us write a custom function for email domain validation.

In [48]:
## import --> FieldValidator, then
## need to mention which specific field
## you want to validate
from pydantic import BaseModel, EmailStr, AnyUrl, Field, field_validator
from typing import List, Dict, Optional, Annotated

## patient data model
class PatientData(BaseModel):

  name: str
  email: EmailStr
  age: int
  weight: float
  married: bool
  allergies: List[str]
  contact_details: Dict[str, str]

  @field_validator('email')
  @classmethod
  def email_validator(cls, value):
    """
    Takes in cls, and input value from user.
    """

    valid_domains = ['hdfc.com', 'icici.com']
    # abc@gmail.com --> only extracting email domain
    domain_name = value.split('@')[-1]

    if domain_name not in valid_domains:
      raise ValueError('Not a valid domain')

    return value

  @field_validator('name')
  @classmethod
  def transform_name(cls, value):
    return value.upper()


def add_patient_data(patient: PatientData):
  print(patient.name)
  print(patient.age)
  print(patient.email)
  print(patient.weight)
  print(patient.married)
  print(patient.allergies)
  print("Data added successfully to the database!")

patient_data = {"name": "Joe",
                "age": 25,
                "weight": 70.5,
                "email": "joecool@hdfc.com",
                "linkedin_url": "https://www.linkedin.com/in/joecool",
                "married": "True",
                "allergies": ["peanuts","shellfish"],
                "contact_details": {"email":"joe@gmail.com","address": "7 hill drive","phone_num":"123-456-7890"}}

## 3. Pass validated model object to functions
patient_1 = PatientData(**patient_data) ## ** will unpack the dict


## test function
add_patient_data(patient_1)


JOE
25
joecool@hdfc.com
70.5
True
['peanuts', 'shellfish']
Data added successfully to the database!


### Summary
- Above we can see that we can create custom `field_validator` functions to validate specific fields on a more granular level.

## Model Validator
- This is how you validate multiple fields that may have multiple conditions dependent upon other fields.
- As an example, lets say that if a patient age is > 60, then they need to have an emergency contact.
- The `@modelvalidator` lets us verify this level of abstraction because we can't put multiple conditions within a field_validator

In [52]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field, field_validator, model_validator
from typing import List, Dict, Optional, Annotated

## patient data model
class PatientData(BaseModel):

  name: str
  email: EmailStr
  age: int
  weight: float
  married: bool
  allergies: List[str]
  contact_details: Dict[str, str]

  @model_validator(mode='after')
  def validate_emergency_contact(cls, model):
    if model.age > 60 and 'emergency' not in model.contact_details:
      raise ValueError('Patients older than 60 must have an emergency contact on file.')
    return model


def add_patient_data(patient: PatientData):
  print(patient.name)
  print(patient.age)
  print(patient.weight)
  print(patient.married)
  print(patient.allergies)
  print("Data added successfully to the database!")



patient_data = {"name": "Joe",
                "age": 70,
                "weight": 70.5,
                "email": "joecool@hdfc.com",
                "linkedin_url": "https://www.linkedin.com/in/joecool",
                "married": "True",
                "allergies": ["peanuts","shellfish"],
                "contact_details": {"email":"joe@gmail.com","address": "7 hill drive","phone_num":"123-456-7890","emergency":"123-344-5678"}}

## 3. Pass validated model object to functions
patient_1 = PatientData(**patient_data) ## ** will unpack the dict


## test function
add_patient_data(patient_1)


Joe
70
70.5
True
['peanuts', 'shellfish']
Data added successfully to the database!


/tmp/ipykernel_3479/2005940556.py:15: PydanticDeprecatedSince212: Using `@model_validator` with mode='after' on a classmethod is deprecated. Instead, use an instance method. See the documentation at https://docs.pydantic.dev/2.13/concepts/validators/#model-after-validator. Deprecated in Pydantic V2.12 to be removed in V3.0.
  @model_validator(mode='after')


## Computed Fields
- Used for specific computations.

In [54]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field, field_validator, model_validator, computed_field
from typing import List, Dict, Optional, Annotated

## patient data model
class PatientData(BaseModel):

  name: str
  email: EmailStr
  age: int
  weight: float
  height: float
  married: bool
  allergies: List[str]
  contact_details: Dict[str, str]

  @computed_field
  @property
  def bmi(self) -> float:
    bmi = round(self.weight/(self.height**2),2)
    return bmi

def add_patient_data(patient: PatientData):
  print(patient.name)
  print(patient.age)
  print(patient.weight)
  print(patient.married)
  print(patient.allergies)
  print("BMI:", patient.bmi)
  print("Data added successfully to the database!")


patient_data = {"name": "Joe",
                "age": 70,
                "weight": 70.5,
                "height": 1.75,
                "email": "joecool@hdfc.com",
                "linkedin_url": "https://www.linkedin.com/in/joecool",
                "married": "True",
                "allergies": ["peanuts","shellfish"],
                "contact_details": {"email":"joe@gmail.com","address": "7 hill drive","phone_num":"123-456-7890","emergency":"123-344-5678"}}

## 3. Pass validated model object to functions
patient_1 = PatientData(**patient_data) ## ** will unpack the dict


## test function
add_patient_data(patient_1)


Joe
70
70.5
True
['peanuts', 'shellfish']
BMI: 23.02
Data added successfully to the database!


## Nested Models

In [58]:
from pydantic import BaseModel

class Address(BaseModel):

  city: str
  state: str
  pin: str

class PatientData(BaseModel):
  name: str
  gender: str
  age: int
  address: Address

## address dict
address_dict = {'city': 'boston', 'state': 'MA', 'pin': '011810'}

address1 = Address(**address_dict)

## patient dict
patient_dict = {'name': 'Dan', 'gender': 'male', 'age': 46, 'address': address1}


## init information
patient1 = PatientData(**patient_dict)

print(patient1)
print(patient1.name)
print(patient1.address)
print(patient1.address.city)

name='Dan' gender='male' age=46 address=Address(city='boston', state='MA', pin='011810')
Dan
city='boston' state='MA' pin='011810'
boston


## Serialization
- Pydantic allows you to serialize and deserialize data

In [61]:
from pydantic import BaseModel

class Address(BaseModel):

  city: str
  state: str
  pin: str

class PatientData(BaseModel):

  name: str
  gender: str
  age: int
  address: Address


## address dict
address_dict = {'city': 'boston', 'state': 'MA', 'pin': '011810'}

address1 = Address(**address_dict)

## patient dict
patient_dict = {'name': 'Dan', 'gender': 'male', 'age': 46, 'address': address1}


## init information
patient1 = PatientData(**patient_dict)


# temp = patient1.model_dump()
temp = patient1.model_dump_json()

print(temp)
print(type(temp))

{"name":"Dan","gender":"male","age":46,"address":{"city":"boston","state":"MA","pin":"011810"}}
<class 'str'>
